# Coleta dos dados da API

In [369]:
import requests
import json

import pandas as pd

## Funções para Query

In [370]:
URL = 'https://graphql.anilist.co'

In [371]:
def queryPages(url:str, query:str, pageAmount:int=1, perPage:int=50):
    pages = []
    for currentPage in range(1, pageAmount+1):

        variables = {
            'page': currentPage,
            'perPage': perPage
        }

        print("Requisitando dados...")
        response = requests.post(url, json={'query': query, 'variables': variables})
        pages.append(json.loads(response.content)["data"]['page'])

    jointPages = []
    key = list(pages[0].keys())[0]
    for page in pages:
        jointPages += page[key]

    return jointPages


def getPages(url:str, query:str, name:str, pageAmount:int=1, perPage:int=50):
    try:
        with open(f'queryCache/{name}.json', 'r') as file:
            pages = json.load(file)
    except FileNotFoundError:
        pages = queryPages(url, query, pageAmount, perPage)
        with open(f'queryCache/{name}.json', 'w') as file:
            file.write(json.dumps(pages, indent=4))
    return pages

In [372]:
def embedPageQuery(query):
    embededQuery =  '''
                    query ($page: Int, $perPage: Int) {
                        page: Page (page: $page, perPage: $perPage){
                    '''
    embededQuery += query
    embededQuery += "}}"
    
    return embededQuery

## Media

In [373]:
mediaQuery = embedPageQuery('''
                media (sort: SCORE_DESC, type: ANIME) {
                    id
                    title {
                        romaji
                        english
                        native
                    }
                    coverImage {
                        large
                    }
                    type
                    format
                    genres
                    status
                    description
                    startDate {
                        year
                        month
                        day
                    }
                    endDate {
                        year
                        month
                        day
                    }
                }
        ''')

In [374]:
pages = getPages(URL, mediaQuery, "media", 5)

In [375]:
df_media = pd.DataFrame(pages)
df_media

,id,title,coverImage,type,format,genres,status,description,startDate,endDate
0,154587,"{'romaji': 'Sousou no Frieren', 'english': 'Fr...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Adventure, Drama, Fantasy]",FINISHED,The adventure is over but life goes on for an ...,"{'year': 2023, 'month': 9, 'day': 29}","{'year': 2024, 'month': 3, 'day': 22}"
1,114129,"{'romaji': 'Gintama: THE FINAL', 'english': 'G...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,Two years have passed following the Tendoshuu'...,"{'year': 2021, 'month': 1, 'day': 8}","{'year': 2021, 'month': 1, 'day': 8}"
2,20996,"{'romaji': 'Gintama°', 'english': 'Gintama Sea...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...","{'year': 2015, 'month': 4, 'day': 8}","{'year': 2016, 'month': 3, 'day': 30}"
3,189046,{'romaji': 'Re:Zero kara Hajimeru Isekai Seika...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Fantasy, Psychologi...",FINISHED,"In the deadly battle at Priestella, Subaru and...","{'year': 2026, 'month': 4, 'day': 8}","{'year': 2026, 'month': 9, 'day': 30}"
4,171627,"{'romaji': 'Chainsaw Man: Reze-hen', 'english'...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Drama, Horror, Romance, Supernatural]",FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...","{'year': 2025, 'month': 9, 'day': 19}","{'year': 2025, 'month': 9, 'day': 19}"
...,...,...,...,...,...,...,...,...,...,...
245,108632,{'romaji': 'Re:Zero kara Hajimeru Isekai Seika...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Fantasy, Psychologi...",FINISHED,"Even after dying countless times, Subaru final...","{'year': 2020, 'month': 7, 'day': 8}","{'year': 2020, 'month': 9, 'day': 30}"
246,109403,"{'romaji': 'Non Non Biyori: Nonstop', 'english...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Comedy, Slice of Life]",FINISHED,Fresh excitement brews as someone new moves in...,"{'year': 2021, 'month': 1, 'day': 11}","{'year': 2021, 'month': 3, 'day': 29}"
247,140960,"{'romaji': 'SPY×FAMILY', 'english': 'SPY x FAM...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Comedy, Slice of Life, Supernatural]",FINISHED,Everyone has a part of themselves they cannot ...,"{'year': 2022, 'month': 4, 'day': 9}","{'year': 2022, 'month': 6, 'day': 25}"
248,161474,{'romaji': 'Seishun Buta Yarou wa Randoseru Gi...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Drama, Psychological, Romance, Supernatural]",FINISHED,"March begins, and there is only one month left...","{'year': 2023, 'month': 12, 'day': 1}","{'year': 2023, 'month': 12, 'day': 1}"


## Estudio

In [376]:
studioQuery =   embedPageQuery('''
                    studios (sort: FAVOURITES_DESC) {
                        id
                        name
                    }
                ''')

In [377]:
studioPages = getPages(URL, studioQuery, 'studio')

In [378]:
df_studio = pd.DataFrame(studioPages)
df_studio

,id,name
0,2,Kyoto Animation
1,11,MADHOUSE
2,569,MAPPA
3,43,ufotable
4,4,bones
5,21,Studio Ghibli
6,858,WIT STUDIO
7,561,A-1 Pictures
8,44,Shaft
9,803,TRIGGER


## Personagem

In [379]:
charQuery = embedPageQuery('''
                characters  (sort: FAVOURITES_DESC) {
                    id
                    name {
                        full
                    }
                    image {
                        large
                    }
                    description
                    gender
                    age
                }
            ''')

In [382]:
charPages = getPages(URL, charQuery, 'character', 5)

Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...


In [383]:
df_character = pd.DataFrame(charPages)
df_character

,id,name,image,description,gender,age
0,127691,{'full': 'Satoru Gojou'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height:__ 190 cm (6'3"")\n__Occupation:__ Juj...",Male,28
1,45627,{'full': 'Levi'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height:__ 160 cm (5'3"")\n\nSquad Captain of ...",Male,NaN
2,40,{'full': 'Luffy Monkey'},{'large': 'https://s4.anilist.co/file/anilistc...,__Height:__ 172 cm\n__Affiliations:__ ~!Straw ...,Male,17-
3,27,{'full': 'Killua Zoldyck'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height:__ 158 cm (5'2"")\n\nKillua is a membe...",Male,11-
4,40882,{'full': 'Eren Yeager'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height:__ 170 cm (5'7"") - 183 cm (6'0"")\n__A...",Male,15-
...,...,...,...,...,...,...
245,157116,{'full': 'Chousou'},{'large': 'https://s4.anilist.co/file/anilistc...,Chousou is half human and half cursed spirit a...,Male,150+
246,46498,{'full': 'Jean Kirstein'},{'large': 'https://s4.anilist.co/file/anilistc...,"__Height__: 175 cm (5'8"")\n__Relatives__: [Mot...",Male,16-
247,67069,{'full': 'Yui Yuigahama'},{'large': 'https://s4.anilist.co/file/anilistc...,"Yui is a cheerful, outgoing girl who becomes t...",Female,16-17
248,19565,{'full': 'Yui Hirasawa'},{'large': 'https://s4.anilist.co/file/anilistc...,__Height:__ 156 cm\n\nYui Hirasawa is one of t...,Female,15-18


## Staff

In [386]:
staffQuery = embedPageQuery('''
                staff (sort: FAVOURITES_DESC) {
                    id
                    image {
                        large
                    }
                    name {
                        full
                    }
                    description
                    gender
                    age
                    primaryOccupations
                }
            ''')

In [389]:
staffPages = getPages(URL, staffQuery, 'staff', 5)

Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...
Requisitando dados...


In [391]:
df_staff = pd.DataFrame(staffPages)
df_staff

,id,image,name,description,gender,age,primaryOccupations
0,95591,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kenjirou Tsuda'},"__Height:__ 170 cm (5'7"")\n__Agency:__ Amuleto...",Male,55.0,[Voice Actor]
1,95118,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Hiroshi Kamiya'},"**Height:** 167 cm (5'6"")\n\n[Agency Profile (...",Male,51.0,[Voice Actor]
2,95869,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Saori Hayami'},__Agency:__ I'm Enterprise\n\n[Profile](https:...,Female,35.0,"[Voice Actor, Vocalist]"
3,95185,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kana Hanazawa'},"__Height:__ 156 cm (5'1"")\n\n[Profile](https:/...",Female,37.0,"[Voice Actor, Vocalist]"
4,95065,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Mamoru Miyano'},"__Height:__ 182 cm (6'0"")\n\n[Website](https:/...",Male,43.0,[Voice Actor]
...,...,...,...,...,...,...,...
245,96882,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Hiro Mashima'},__[Twitter](https://twitter.com/hiro_mashima)_...,Male,49.0,"[Mangaka, Artist]"
246,118475,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Yoshino Aoyama'},__Nickname:__ Yoppii (よっぴー)\n\n[Profile (81pro...,Female,30.0,[Voice Actor]
247,108173,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Mengo Yokoyari'},"__Favorite WSJ Manga:__ Chainsaw-man, Boku no ...",Female,38.0,[Mangaka]
248,126963,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Sayumi Suzushiro'},[Profile (Arts Vision)](https://www.artsvision...,Female,28.0,[Voice Actor]


## Usuarios

In [392]:
userQuery = embedPageQuery('''
                users (sort: WATCHED_TIME_DESC){
                    id
                    name
                    avatar {
                        large
                    }
                    createdAt
                }
            ''')

In [393]:
userPages = getPages(URL, userQuery, 'staff', 5)

In [394]:
df_user = pd.DataFrame(userPages)
df_user

,id,image,name,description,gender,age,primaryOccupations
0,95591,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kenjirou Tsuda'},"__Height:__ 170 cm (5'7"")\n__Agency:__ Amuleto...",Male,55.0,[Voice Actor]
1,95118,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Hiroshi Kamiya'},"**Height:** 167 cm (5'6"")\n\n[Agency Profile (...",Male,51.0,[Voice Actor]
2,95869,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Saori Hayami'},__Agency:__ I'm Enterprise\n\n[Profile](https:...,Female,35.0,"[Voice Actor, Vocalist]"
3,95185,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Kana Hanazawa'},"__Height:__ 156 cm (5'1"")\n\n[Profile](https:/...",Female,37.0,"[Voice Actor, Vocalist]"
4,95065,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Mamoru Miyano'},"__Height:__ 182 cm (6'0"")\n\n[Website](https:/...",Male,43.0,[Voice Actor]
...,...,...,...,...,...,...,...
245,96882,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Hiro Mashima'},__[Twitter](https://twitter.com/hiro_mashima)_...,Male,49.0,"[Mangaka, Artist]"
246,118475,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Yoshino Aoyama'},__Nickname:__ Yoppii (よっぴー)\n\n[Profile (81pro...,Female,30.0,[Voice Actor]
247,108173,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Mengo Yokoyari'},"__Favorite WSJ Manga:__ Chainsaw-man, Boku no ...",Female,38.0,[Mangaka]
248,126963,{'large': 'https://s4.anilist.co/file/anilistc...,{'full': 'Sayumi Suzushiro'},[Profile (Arts Vision)](https://www.artsvision...,Female,28.0,[Voice Actor]
